## Analysis

In [ ]:
import pandas as pd
from pathlib import Path
from plots import plot_compare
from ml import calculate_residuals
from run_surrogate import load_graph_data, get_device, load_trained_model, run_all_predictions
from animate import animate_residuals
from run_surrogate import bootstrap_simulation

# Case Parameters
SIM_NAME = "example-model"
CASE = "case0"

# Pathing
ROOT = Path().resolve().parent
BASE = ROOT / "sims" / SIM_NAME / "training_data" / "graph" / CASE
MODEL_PATH =  BASE / "model_graphsage.pt"

# Animation Parameters
FPS = 20
CMAP = "turbo"
OFF_SCREEN = True
WINDOW_SIZE = (1920, 1088)

#TODO: Add heatmap (u vs p vs rmse) averaged over all epochs

#### Load Simulation Results (Truth)

In [ ]:
# Load PyFR simulation data
results_path = ROOT / "sims" / SIM_NAME / "training_data" / f"{CASE}-results.csv"

df_sim = pd.read_csv(results_path)
df_sim = df_sim[df_sim['step'] > 0] # First time-step is initial condition for surrogate
df_sim.head(2)

,step,node_id,n_x,n_y,p,u,v,vn
18408,1,0,-0.500000,0.00000,3.453429,-0.047711,-0.000553,0.047714
18409,1,1,-0.487464,0.11126,3.354103,-0.054916,-0.011371,0.056081


#### Next-Step Prediction Test

Perform teacher-forced evaluation where the model iteratively predict the next 
state of each node using ground-truth (simulated) values as input at each step. 
This kind of run evaluates the model's "next-step" predictive accuracy without 
compounding its own errors and writes out node-level predictions for each step 
in a CSV with the standard schema for further analysis.

In [ ]:
# Loading GNN model
data = load_graph_data(SIM_NAME, CASE)
device = get_device()
model = load_trained_model(MODEL_PATH, device=device)

In [ ]:
# Run surrogate model
steps = df_sim['step'].max()  # Use the same number of steps as simulation
run_all_predictions(model, data, steps=200, start_step=0)

In [ ]:
# Load surrogate prediction results
out_dir = ROOT / "sims" / SIM_NAME / "training_data" / "rollouts"
out_csv_path = out_dir / f"{CASE}-teacher_forced.csv"
df_ml = pd.read_csv(out_csv_path)
df_ml.head(2)

In [ ]:
# Quick checks
assert len(df_ml) != 0, "Missing surrogate data!"
assert len(df_sim) != 0, "Missing sim data!"
assert len(df_ml) == len(df_sim), f"Should be equal: {len(df_ml)}, {len(df_sim)}"

In [ ]:
# Calculate residuals at each node
df_res = calculate_residuals(df_sim, df_ml)
df_res.describe()

In [ ]:
# Quick checks
assert len(df_ml) == len(df_sim) == len(df_res), f"Should be equal: {len(df_ml)}, {len(df_sim)}, {len(df_res)}"

In [ ]:
"""
[ Nodal Residual Analysis ]

This section analyses how well the surrogate (ML) model predicts the simulation 
at each node and time step.

The plot generated below shows, for a selected time step (STEP), a comparison 
between the simulation ("ground truth"), the surrogate model's prediction, and 
the residual (error) between them, at all nodes. 

The visualisation helps diagnose:
- Where in the domain the model performs well or poorly,
- Which regions tend to have larger errors,
- And whether the errors display any spatial pattern or structure.
"""

# Plot single step: 
STEP = 50 # <-- manually select

fig = plot_compare(df_sim, df_ml, df_res, metric="p", step=STEP)
fig.show()

In [ ]:
"""
If enabled, create an animation of the residuals between simulation and 
surrogate model predictions at all time steps
"""
ANIMATE = False

if ANIMATE:
    animate_residuals(df_sim, df_ml, df_res, SIM_NAME, 
                      dir_name="next-step-residuals")

#### Boostrap Analaysis

In [ ]:
"""
[ Bootstrap Model Description ]

The bootstrap model is a variant of the surrogate (ML) model designed to assess 
the stability and predictive reliability of the surrogate on its own outputs. 
In the bootstrap procedure, the surrogate model is used recursively and 
predictions at one time step are used as inputs to predict the next time step, 
simulating a full rollout where only the surrogate's outputs are used 
(without correction from the 'ground truth' simulation data).

This approach evaluates how errors may accumulate over time when the surrogate 
is applied in a self-consistent way, mirroring how it would be used in practice 
during deployment. It also highlights any drift, instability, or significant 
error amplification present when the surrogate is run in closed-loop fashion.

NOTE: Comparing the surrogate solely via step-by-step residuals can be 
misleading. Should consider statistical evaluation methods (e.g. RMSE 
distributions) to better assess surrogate performance over time.
"""
br_path = bootstrap_simulation(SIM_NAME, CASE, steps=200, start_step=0)

In [4]:
# Load bootstrap results
out_dir = ROOT / "sims" / SIM_NAME / "training_data" / "rollouts"
out_csv_path = out_dir / f"{CASE}-bootstrap.csv"
df_boot = pd.read_csv(out_csv_path)
df_boot.head(2)

,step,node_id,n_x,n_y,p,u,v,vn
0,1,0,-0.500000,0.00000,1.373777,0.911101,0.014114,0.911210
1,1,1,-0.487464,0.11126,1.367697,0.911102,0.014324,0.911215


In [5]:
# Calculate residuals
df_boot_res = calculate_residuals(df_sim, df_boot)
df_boot_res.describe()

,node_id,n_x,n_y,p,u,v,vn,step
count,3.681600e+06,3.681600e+06,3.681600e+06,3.681600e+06,3.681600e+06,3.681600e+06,3.681600e+06,3.681600e+06
mean,9.203500e+03,1.056923e+01,2.353478e-02,1.641786e+00,8.364399e+01,3.677666e+05,2.095885e+01,1.005000e+02
std,5.313933e+03,1.091237e+01,3.196470e+00,8.486327e+00,3.265799e+04,1.028295e+08,1.358423e+03,5.773431e+01
min,0.000000e+00,-8.000000e+00,-8.000000e+00,2.151899e-16,0.000000e+00,2.370861e-06,1.438709e-08,1.000000e+00
25%,4.601750e+03,7.516849e-01,-2.297408e+00,1.247166e-01,8.860484e-02,9.801569e+00,1.115199e-01,5.075000e+01
50%,9.203500e+03,8.957236e+00,3.747690e-02,3.861383e-01,3.497652e-01,2.720853e+02,5.312239e-01,1.005000e+02
75%,1.380525e+04,1.935340e+01,2.351493e+00,1.725112e+00,1.541909e+00,1.045790e+04,3.041995e+00,1.502500e+02
max,1.840700e+04,3.500000e+01,8.000000e+00,9.674594e+03,5.979349e+07,1.568204e+11,1.236661e+06,2.000000e+02


In [ ]:
# Quick checks
assert len(df_boot) == len(df_sim) == len(df_boot_res), f"Should be equal: {len(df_boot)}, {len(df_sim)}, {len(df_boot_res)}"

In [ ]:
"""
Animate bootstrap residual plots

If enabled, create an animation of the residuals between simulation and 
surrogate model predictions at all time steps
"""
ANIMATE = False

if ANIMATE:
    animate_residuals(df_sim, df_boot, df_boot_res, SIM_NAME, 
                      dir_name="boot-residuals")

100%|██████████| 199/199 [00:17<00:00, 11.43it/s]


Animation saved to /Volumes/connor/dev/projects/ns2d-surrogate/sims/example-model/animations/example-model-residual-plots.mp4
